# Lab 05: Joins & Set Operations

## Objectives
- Perform inner, outer, left, right, semi, and anti joins
- Understand when each join type is appropriate
- Use set operations: union, intersect, except
- Join NYC Taxi data with GitHub Archive data

## Exam Domain
- **Developing DataFrame/DataSet API Applications — 30%**

## Estimates
- **Time:** ~35 minutes
- **Cost:** ~$1 (serverless)
- **Compute:** Serverless

![Join Types](../assets/diagrams/lab-05-join-types.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

taxi_df = spark.table("taxi_trips")
github_df = spark.table("github_events")

## A. Preparing Join-Ready DataFrames

To demonstrate joins clearly, we'll create two smaller DataFrames with a shared key: the hour of day. This lets us join taxi trips with GitHub events that happened in the same hour.

In [ ]:
from pyspark.sql.functions import col, hour, count, avg, round, date_format

# Taxi trips aggregated by hour of day
taxi_by_hour = (
    taxi_df
    .withColumn("hour_of_day", hour("tpep_pickup_datetime"))
    .groupBy("hour_of_day")
    .agg(
        count("*").alias("taxi_trips"),
        round(avg("total_amount"), 2).alias("avg_fare"),
    )
)
taxi_by_hour.orderBy("hour_of_day").show()

In [ ]:
from pyspark.sql.functions import to_timestamp

# GitHub events aggregated by hour of day
github_by_hour = (
    github_df
    .withColumn("event_hour", hour(to_timestamp("created_at")))
    .groupBy("event_hour")
    .agg(
        count("*").alias("github_events"),
    )
    .withColumnRenamed("event_hour", "hour_of_day")
)
github_by_hour.orderBy("hour_of_day").show()

## B. Inner Join

Returns only rows where the key exists in BOTH DataFrames. This is the default join type.

In [ ]:
# Inner join — only hours present in both datasets
inner = taxi_by_hour.join(github_by_hour, on="hour_of_day", how="inner")
print(f"Inner join rows: {inner.count()}")
inner.orderBy("hour_of_day").show()

## C. Left, Right, and Full Outer Joins

In [ ]:
# Left join — all rows from taxi, matching rows from github (null if no match)
left = taxi_by_hour.join(github_by_hour, on="hour_of_day", how="left")
print(f"Left join rows: {left.count()}")
left.orderBy("hour_of_day").show()

In [ ]:
# Right join — all rows from github, matching rows from taxi
right = taxi_by_hour.join(github_by_hour, on="hour_of_day", how="right")
print(f"Right join rows: {right.count()}")
right.orderBy("hour_of_day").show()

In [ ]:
# Full outer join — all rows from both, nulls where no match
full = taxi_by_hour.join(github_by_hour, on="hour_of_day", how="full")
print(f"Full outer join rows: {full.count()}")
full.orderBy("hour_of_day").show()

## D. Semi and Anti Joins

Semi and anti joins are filtering operations — they don't add columns from the right DataFrame.

- **Left Semi:** Keep left rows that HAVE a match in the right
- **Left Anti:** Keep left rows that DO NOT have a match in the right

In [ ]:
# Left semi join — taxi hours that also appear in github
semi = taxi_by_hour.join(github_by_hour, on="hour_of_day", how="left_semi")
print(f"Semi join rows: {semi.count()}")
print(f"Columns: {semi.columns}")  # Only left columns
semi.show()

In [ ]:
# Left anti join — taxi hours that do NOT appear in github
anti = taxi_by_hour.join(github_by_hour, on="hour_of_day", how="left_anti")
print(f"Anti join rows: {anti.count()}")
anti.show()

> **Exam Tip:** Semi and anti joins are common exam topics. Key distinction:
> - Semi join = `WHERE EXISTS` in SQL (keep matching rows, no extra columns)
> - Anti join = `WHERE NOT EXISTS` in SQL (keep non-matching rows)
> - Neither adds columns from the right DataFrame.

## E. Join with Different Column Names

When the join key has different names in each DataFrame, use a column expression instead of a string.

In [ ]:
# Rename the key column to simulate different names
github_renamed = github_by_hour.withColumnRenamed("hour_of_day", "event_hour")

# Join on different column names using expression
joined = taxi_by_hour.join(
    github_renamed,
    taxi_by_hour.hour_of_day == github_renamed.event_hour,
    how="inner"
)
joined.show(5)

## F. Set Operations

Set operations combine two DataFrames that have the same schema (same columns and types).

| Operation | SQL Equivalent | Behavior |
|-----------|---------------|----------|
| `union()` / `unionAll()` | `UNION ALL` | All rows from both (keeps duplicates) |
| `unionByName()` | `UNION ALL` | Matches by column name, not position |
| `intersect()` | `INTERSECT` | Rows present in both |
| `exceptAll()` | `EXCEPT ALL` | Rows in left but not in right |

In [ ]:
# Split taxi data into two halves for set operations
from pyspark.sql.functions import expr

taxi_half1 = taxi_df.limit(1000)
taxi_half2 = taxi_df.limit(1500)  # overlaps with half1

# Union — all rows (with duplicates)
unioned = taxi_half1.union(taxi_half2)
print(f"Half1: {taxi_half1.count()}, Half2: {taxi_half2.count()}, Union: {unioned.count()}")

# Intersect — common rows
common = taxi_half1.intersect(taxi_half2)
print(f"Intersect: {common.count()}")

# Except — rows in half1 but not in half2
diff = taxi_half1.exceptAll(taxi_half2)
print(f"Except: {diff.count()}")

### SQL Alternative

In [ ]:
taxi_by_hour.createOrReplaceTempView("taxi_hours")
github_by_hour.createOrReplaceTempView("github_hours")

# SQL inner join
spark.sql("""
    SELECT t.hour_of_day, t.taxi_trips, t.avg_fare, g.github_events
    FROM taxi_hours t
    INNER JOIN github_hours g ON t.hour_of_day = g.hour_of_day
    ORDER BY t.hour_of_day
""").show()

# SQL anti join
spark.sql("""
    SELECT t.*
    FROM taxi_hours t
    LEFT ANTI JOIN github_hours g ON t.hour_of_day = g.hour_of_day
""").show()

## Exam-Style Review

**Q1.** What is the default join type when you call `df1.join(df2, on="key")`?
- A) left
- B) inner
- C) full
- D) cross

**Q2.** What columns are included in the result of a left semi join?
- A) All columns from both DataFrames
- B) All columns from the left DataFrame only
- C) All columns from the right DataFrame only
- D) Only the join key column

**Q3.** What is the difference between `union()` and `unionByName()`?
- A) `union()` matches columns by name; `unionByName()` matches by position
- B) `union()` matches columns by position; `unionByName()` matches by name
- C) `union()` removes duplicates; `unionByName()` keeps duplicates
- D) They are identical

**Q4.** You have two DataFrames with the same schema. You want to find rows in df1 that do NOT exist in df2. Which operation should you use?
- A) `df1.join(df2, how="left_anti")`
- B) `df1.join(df2, how="left_semi")`
- C) `df1.intersect(df2)`
- D) `df1.union(df2)`

### Answers
- **Q1: B** — The default join type is `inner`.
- **Q2: B** — Semi joins return only columns from the left DataFrame. They are a filtering operation.
- **Q3: B** — `union()` matches by column position (order matters). `unionByName()` matches by column name (order doesn't matter).
- **Q4: A** — `left_anti` returns rows from df1 that have no match in df2. `exceptAll()` also works for same-schema DataFrames.

## Key Takeaways
- Inner join is the default; always specify the join type explicitly for clarity
- Semi and anti joins are filter operations — they don't add columns from the right side
- Semi = keep matches; anti = keep non-matches
- `union()` matches by position; `unionByName()` matches by column name
- `intersect()` finds common rows; `exceptAll()` finds rows only in the left

**Next:** [Lab 06 — Complex Data Types & JSON](06-complex-data-types-json.ipynb)

In [ ]:
spark.catalog.dropTempView("taxi_hours")
spark.catalog.dropTempView("github_hours")
print("Lab 05 cleanup complete.")